# <font color='#1B3A5C'>Auditoría de Calidad de Datos: Consumo Eléctrico por Código Postal</font>

> Antes de confiar en el modelado (notebooks 03 a 06), se audita la integridad del dataset de consumo eléctrico por código postal. El hallazgo central: el consumo por código postal sufre quiebres artificiales y abruptos en 2024-2025 (colapsos e inflaciones), compatibles con reasignaciones de grandes puntos de suministro entre códigos, que contaminan la evaluación de los modelos. La redistribución es solo parcial y coexiste con un declive neto, no una reasignación limpia barrio a barrio.

### Qué hace este notebook
1. Síntoma: muchas series saltan de nivel o se desploman en 2025.
2. ¿Faltan datos o se redistribuyen? El consumo total de la ciudad.
3. ¿Es la fuente o el ETL? Verificación contra los datos crudos de OpenData.
4. Clasificación: auditoría sistemática, CPs limpios vs sucios.
5. Volatilidad: degradación sistémica de 2025.
6. ¿Adónde se fue el consumo? Análisis geográfico (conservación + mapa).
7. Conclusiones: mecanismo, lista de exclusión y limitaciones para la memoria.

> Resultado: una lista de 12 CPs a excluir de la evaluación (no del entrenamiento) y la justificación documentada de por qué el periodo 2025 pone un techo al R² alcanzable.

## Librerías

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pymongo import MongoClient
from math import radians, sin, cos, asin, sqrt
import warnings; warnings.filterwarnings('ignore')

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#D3D3D3'; plt.rcParams['grid.linewidth'] = 0.4
plt.rcParams['axes.spines.top'] = False; plt.rcParams['axes.spines.right'] = False

# Paleta (consistente con los notebooks 03-06)
C1='#264653'; C2='#2A9D8F'; C3='#E9C46A'; C4='#F4A261'; C5='#E76F51'; C6='#A8DADC'
TITULO='#1B3A5C'; SUBTITULO='#C0392B'

---
## Carga de datos

> Se parte de `dataset_features` (consumo + centroides por CP). Se construye un diccionario `series` con la serie de `mwh_total` de cada barrio (frecuencia 6h) y se guardan los centroides para el análisis geográfico.

In [ ]:
client = MongoClient('mongodb://mongo:27017/')
db = client['tfm_energy']

docs = list(db['dataset_features'].find({}, {'_id':0,'datetime':1,'cod_postal':1,'mwh_total':1,
                                             'centroide_lat':1,'centroide_lon':1}))
df = pl.DataFrame(docs)
CPS_TODOS = sorted(df['cod_postal'].unique().to_list())
FIN_TRAIN = '2024-12-31 18:00:00'   # fin de train
FIN_VAL   = '2025-09-30 18:00:00'   # fin de validacion -> a su derecha, el test (oct-nov 2025)

# Serie de cada CP (pandas, 6h)
series = {}
for cp in CPS_TODOS:
    s = (df.filter(pl.col('cod_postal')==cp).sort('datetime')
           .select(['datetime','mwh_total']).to_pandas().set_index('datetime')['mwh_total'])
    series[cp] = s.asfreq('6h')

# Centroide de cada CP (para el analisis geografico)
cent = {}
for cp in CPS_TODOS:
    r = df.filter(pl.col('cod_postal')==cp).select(['centroide_lat','centroide_lon']).row(0)
    cent[cp] = (r[0], r[1])

print(f"CPs: {len(CPS_TODOS)} | desde {df['datetime'].min()} hasta {df['datetime'].max()}")

---
# <font color='#1B3A5C'>1. El síntoma: series que se rompen en 2025</font>

> Al revisar las 42 series, muchas presentan en 2024-2025 quiebres bruscos de nivel, picos sostenidos o caídas a casi cero, justo en el tramo de validación/test. La línea amarilla marca el fin de train y la roja el fin de validación (a su derecha, el test).

In [ ]:
fig, axes = plt.subplots(7, 6, figsize=(22, 20), sharex=True)
for ax, cp in zip(axes.ravel(), CPS_TODOS):
    s = series[cp]
    ax.plot(s.index, s.values, color=C1, lw=0.5)
    ax.axvline(pd.Timestamp(FIN_TRAIN), color=C3, ls='--', lw=0.7)
    ax.axvline(pd.Timestamp(FIN_VAL),   color=C5, ls='--', lw=0.7)
    ax.set_title(cp, fontsize=9, color=TITULO); ax.tick_params(labelsize=6)
fig.suptitle('mwh_total por código postal (2019-2025) · amarillo=fin train, rojo=fin val',
             fontsize=15, fontweight='bold', color=TITULO)
plt.tight_layout(); plt.show()

---
# <font color='#1B3A5C'>2. ¿Faltan datos o se redistribuyen?</font>

> Primera pregunta clave: cuando un barrio se desploma, ¿desaparece el consumo (datos incompletos) o se recoloca en otro barrio? Para responderlo se mira el consumo total de Barcelona (suma de los 42 CPs) mes a mes. Si el total se mantiene, no faltan datos a nivel ciudad.

In [ ]:
tot = None
for cp in CPS_TODOS:
    s = series[cp].resample('MS').sum()
    tot = s if tot is None else tot.add(s, fill_value=0)

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(tot.index, tot.values, color=C1, lw=1.5, marker='o', ms=3)
ax.axvline(pd.Timestamp(FIN_VAL), color=C5, ls='--', lw=1)
ax.set_title('Consumo TOTAL de Barcelona por mes (suma 42 CPs)', fontweight='bold', color=TITULO)
ax.set_ylabel('MWh/mes'); plt.tight_layout(); plt.show()

> El total de la ciudad se mantiene dentro de su rango histórico todo el periodo (no se desploma), pero los meses de 2025 están en la parte baja del rango.

 - No faltan datos de forma masiva a nivel ciudad: el agregado se conserva.
 - Sí hay un declive neto en 2025 (se cuantifica en la sección 6).
 - La mayoría de los quiebres por barrio se compensan entre sí, lo que apunta a redistribución más que a pérdida de datos.

---
# <font color='#1B3A5C'>3. ¿Es la fuente o nuestro ETL?</font>

> Antes de culpar al pipeline, se verifica el caso más roto (08009) contra la colección `raw_electricity`, los datos crudos de OpenData antes de cualquier transformación. Si el quiebre ya está ahí, el artefacto es de la fuente, no del ETL.

In [ ]:
raw = list(db['raw_electricity'].find({'Codi_Postal': 8009}, {'_id':0,'Data':1,'Valor':1}))
rdf = (pl.DataFrame(raw)
       .with_columns(pl.col('Data').str.slice(0, 7).alias('mes'))
       .group_by('mes').agg(pl.col('Valor').sum().alias('suma_valor'), pl.len().alias('n_filas'))
       .sort('mes'))
print('08009 en el CRUDO de OpenData (suma mensual de Valor):')
print('Normal 2019-2022: ~5-7 M/mes con ~460 filas')
print(rdf.filter(pl.col('mes').is_in(['2023-09','2023-10','2023-11','2025-09','2025-10','2025-11'])))

> El quiebre ya está en los datos crudos de OpenData, no lo introduce el ETL.

 - El pico de 2023-10 (≈22,5 M, 4× lo normal) y el colapso de 2025-10 (≈0,4 M, <10%) ya están en el CSV crudo de OpenData.
 - `n_filas` se mantiene en ~460/mes incluso en los meses colapsados: la estructura de registros está completa, lo que cambia son los valores.
 - El ingester (`electricity_od_bcn.py`) es un passthrough: coge `Codi_Postal` tal cual de la fuente (solo `.zfill(5)`), sin geocodificación ni atribución. El artefacto es inherente a OpenData, no del pipeline.

---
# <font color='#1B3A5C'>4. Clasificación: limpios vs sucios</font>

> Auditoría sistemática de los 42 CPs. Para cada barrio se compara cada mes de 2025 con su mismo mes de 2019-2022 (controla la estacionalidad), separando validación (ene-sep) y test (oct-nov).

 - r_val / r_test: mediana del cociente (mes 2025 entre el mismo mes 2019-22). ≈1 sano, <0,55 colapso, >1,7 inflado.
 - infl_max: pico máximo (mes entre base), detecta inflación.
 - pct0_test: % de bloques casi-cero en test, detecta reatribución a cero.

In [ ]:
mdf = df.with_columns(pl.col('datetime').dt.year().alias('y'), pl.col('datetime').dt.month().alias('m'))
mm = mdf.group_by(['cod_postal','y','m']).agg(pl.col('mwh_total').mean().alias('mean_mes'))
basem = mm.filter(pl.col('y')<=2022).group_by(['cod_postal','m']).agg(pl.col('mean_mes').mean().alias('base_m'))
mm = mm.join(basem, on=['cod_postal','m']).with_columns((pl.col('mean_mes')/pl.col('base_m')).alias('ratio'))

def med_ratio(months):
    return (mm.filter((pl.col('y')==2025) & (pl.col('m').is_in(months)))
              .group_by('cod_postal').agg(pl.col('ratio').median().round(2)))
r_val  = med_ratio([1,2,3,4,5,6,7,8,9]).rename({'ratio':'r_val'})
r_test = med_ratio([10,11]).rename({'ratio':'r_test'})
infl   = mm.filter(pl.col('y')>=2023).group_by('cod_postal').agg(pl.col('ratio').max().round(1).alias('infl_max'))

basey = mdf.filter(pl.col('y')<=2022).group_by('cod_postal').agg(pl.col('mwh_total').median().alias('by'))
dfb   = mdf.join(basey, on='cod_postal').with_columns((pl.col('mwh_total') < 0.1*pl.col('by')).alias('n0'))
pct0  = (dfb.filter((pl.col('y')==2025)&(pl.col('m').is_in([10,11])))
            .group_by('cod_postal').agg((pl.col('n0').mean()*100).round(0).alias('pct0_test')))

audit = (r_val.join(r_test,on='cod_postal').join(infl,on='cod_postal').join(pct0,on='cod_postal'))
audit = audit.with_columns(((pl.col('r_test')<0.55)|(pl.col('r_test')>1.7)|
                            (pl.col('r_val')<0.55)|(pl.col('r_val')>1.7)|
                            (pl.col('infl_max')>3)|(pl.col('pct0_test')>15)).alias('EXCLUIR'))
audit = audit.sort(['EXCLUIR','r_test'], descending=[True, False])
with pl.Config(tbl_rows=50):
    print(audit)
print('EXCLUIR (por datos):', audit.filter(pl.col('EXCLUIR'))['cod_postal'].to_list())

### <font color='#C0392B'>4.1 Los sucios, por gravedad</font>

> Los 11 marcados por la auditoría más el 08037 (target imputado en el EDA, motivo aparte) suman 12 CPs a excluir de la evaluación. Ordenados por severidad:

 - Tier A, reatribución a casi cero en test (inequívoco): 08011 (r_test 0,05, 88% bloques ~0), 08009 (0,08, 75%), 08007 (0,10, 36%). El consumo se fue a otros barrios.
 - Tier B, test colapsado a <0,5: 08013 (0,35), 08010 (0,40), 08006 (0,41 con inflación 3,3×), 08005 (0,50). Validación sana, pero el test cae a un tercio o la mitad.
 - Tier C, validación corrupta: 08008 (r_val 0,43), 08019 (0,51), 08036 (1,86, inflado ~2×), 08026 (r_test 1,85, inflado).

> 08037 se excluye por un motivo distinto: su target jul-nov 2025 fue imputado en el EDA (sintético), no es demanda real.

In [ ]:
# Lista oficial de exclusion (solo de la EVALUACION, no del entrenamiento)
CPS_SUCIOS  = ['08011','08009','08007','08013','08010','08006','08005',
               '08019','08008','08036','08026','08037']
CPS_LIMPIOS = [cp for cp in CPS_TODOS if cp not in CPS_SUCIOS]
print(f"SUCIOS (excluidos de metricas): {len(CPS_SUCIOS)}")
print(f"LIMPIOS (usados en metricas)  : {len(CPS_LIMPIOS)}")
print(CPS_LIMPIOS)

In [ ]:
# Vista individual de los SUCIOS (zona sombreada = test): se ve el colapso / la inflacion
def plot_grupo(cps, etiqueta):
    print('='*80 + f'\n  {etiqueta}  ({len(cps)} CPs)\n' + '='*80)
    for cp in cps:
        s = series[cp]; med = float(np.nanmedian(s.values))
        fig, ax = plt.subplots(figsize=(15, 2.6))
        ax.plot(s.index, s.values, color=C1, lw=0.6)
        ax.axhline(med, color=C2, lw=0.8, ls=':')
        ax.axvline(pd.Timestamp(FIN_TRAIN), color=C3, ls='--', lw=0.8)
        ax.axvline(pd.Timestamp(FIN_VAL),   color=C5, ls='--', lw=0.8)
        ax.axvspan(pd.Timestamp(FIN_VAL), s.index.max(), color=C5, alpha=0.07)
        ax.set_title(f'{cp} · mediana = {med:,.0f} MWh', fontsize=11, color=TITULO, fontweight='bold')
        ax.set_ylabel('MWh')
        plt.tight_layout(); plt.show()

plot_grupo(CPS_SUCIOS, 'CPs SUCIOS — excluidos de la evaluación')

---
# <font color='#1B3A5C'>5. Volatilidad: degradación sistémica de 2025</font>

> Más allá de los 12 rotos, ¿está el resto del dataset afectado? Se compara el coeficiente de variación (CV = desviación/media, libre de escala) de 2025 contra 2019-2022. Un salto alto significa que la serie se volvió más ruidosa.

In [ ]:
def cv(filt):
    return (mdf.filter(filt).group_by('cod_postal')
            .agg((pl.col('mwh_total').std()/pl.col('mwh_total').mean()).alias('cv')))
cv_base = cv(pl.col('y')<=2022).rename({'cv':'cv_1922'})
cv_25   = cv(pl.col('y')==2025).rename({'cv':'cv_2025'})
vol = (cv_base.join(cv_25, on='cod_postal')
       .with_columns((pl.col('cv_2025')/pl.col('cv_1922')).round(2).alias('salto_vol'),
                     pl.col('cv_1922').round(2), pl.col('cv_2025').round(2))
       .with_columns(pl.col('cod_postal').is_in(CPS_SUCIOS).alias('ya_sucio'))
       .sort('salto_vol', descending=True))
with pl.Config(tbl_rows=50):
    print(vol)

> Ruido no es corrupción. Casi la mitad de los CPs tienen `salto_vol` en 1,3-1,8: en 2025 se volvieron más ruidosos aunque su nivel se mantenga. Es un artefacto sistémico de la fuente (reatribución fina mes a mes), distinto del colapso de los 12.

 - Decisión metodológica: estos barrios ruidosos NO se excluyen. Su consumo es real, solo más difícil de predecir; quitarlos sería cherry-picking para inflar el R².
 - Se mantienen y se reporta el R² honesto. La consecuencia: el periodo 2025 pone un techo al R² alcanzable por cualquier modelo. No es un fallo del modelado, es la calidad del dato.

---
# <font color='#1B3A5C'>6. ¿Adónde se fue el consumo? Análisis geográfico</font>

> Si el consumo se redistribuye, debería reaparecer en otros barrios. Se calcula el delta de cada CP (media 2025 menos media 2019-2022) y se comprueba la conservación, los vecinos y un mapa.

In [ ]:
delta = {cp: series[cp].loc['2025'].mean() - series[cp].loc['2019':'2022'].mean() for cp in CPS_TODOS}
d = pd.Series(delta).sort_values()
print(f"CONSERVACIÓN: suma de deltas = {d.sum():,.0f} MWh/bloque (≈0 si solo se recolocó)")
print(f"Pérdida total (los que bajan): {d[d<0].sum():,.0f}")
print(f"Ganancia total (los que suben): {d[d>0].sum():,.0f}")
print('\n── Mayores PÉRDIDAS ──'); print(d.head(8).round(0))
print('\n── Mayores GANANCIAS ──'); print(d.tail(8).round(0))

In [ ]:
# Vecinos: cuando un CP colapsa, sus vecinos cercanos, ganaron o perdieron?
def km(a, b):
    la1,lo1,la2,lo2 = map(radians, [*a, *b])
    h = sin((la2-la1)/2)**2 + cos(la1)*cos(la2)*sin((lo2-lo1)/2)**2
    return 6371*2*asin(sqrt(h))

for cp in ['08009','08011','08007','08013']:
    vec = sorted((km(cent[cp], cent[o]), o) for o in CPS_TODOS if o != cp)[:5]
    print(f"\n{cp} (Δ={delta[cp]:,.0f}) ── 5 vecinos más cercanos:")
    for dkm, o in vec:
        signo = '↑ ganó' if delta[o] > 0 else '↓ perdió'
        print(f"   {o}  a {dkm:4.1f} km   Δ={delta[o]:>10,.0f}  {signo}")

In [ ]:
# Mapa: rojo = perdió, verde = ganó
lats = [cent[cp][0] for cp in CPS_TODOS]; lons = [cent[cp][1] for cp in CPS_TODOS]
vals = [delta[cp] for cp in CPS_TODOS]
fig, ax = plt.subplots(figsize=(9, 8))
sc = ax.scatter(lons, lats, c=vals, cmap='RdYlGn',
                s=[max(30, abs(v)/60) for v in vals], edgecolors='k', linewidths=0.5)
for cp in CPS_TODOS:
    ax.annotate(cp[-3:], (cent[cp][1], cent[cp][0]), fontsize=6, ha='center', va='center')
plt.colorbar(sc, label='Δ consumo 2025 vs 2019-22 (MWh/bloque)')
ax.set_title('¿Adónde se fue el consumo?  (rojo = perdió · verde = ganó)',
             fontweight='bold', color=TITULO)
ax.set_xlabel('longitud'); ax.set_ylabel('latitud')
plt.tight_layout(); plt.show()

> La conservación es PARCIAL. Las ganancias compensan solo ~35% de las pérdidas, así que hay un declive neto (~10%) además de la redistribución.

 - El test de vecinos muestra que los barrios que caen están rodeados de barrios que también caen (Eixample central).
 - No es una reatribución limpia barrio a barrio, sino una caída agrupada de toda una zona densa, solo parcialmente concentrada en algún CP (08036, el mayor ganador, pegado al cluster rojo).

---
# <font color='#1B3A5C'>7. Conclusiones: mecanismo y limitaciones</font>

### El mecanismo
> El consumo de un CP está dominado por unos pocos puntos de suministro grandes (CUPS), y el dato de OpenData llega agregado por código postal desde la distribuidora vía Datadis. Los quiebres son bruscos y escalonados (no graduales como la demanda real), lo que descarta una variación de demanda y apunta a un artefacto de atribución o reporte. Caben dos mecanismos no excluyentes:

 - Reasignación de CUPS entre códigos: un cambio en la geocodificación de la distribuidora (~2023-2024, de dirección de facturación a ubicación física) desplaza de golpe el consumo de un gran suministro de un código a otro. Predice conservación: lo que pierde un código lo gana otro.
 - Error de agregación o reporte en el paso CUPS a código postal: suministros que se pierden, duplican o se asignan mal al construir el agregado por código. Predice pérdida neta, sin reaparición en otro barrio.

> La evidencia apunta a una mezcla: la conservación es solo parcial (~35%) y hay un declive neto (~10%), más consistente con que parte del consumo no se reasigna sino que se pierde en la agregación. No es, por tanto, una reasignación limpia barrio a barrio.

> Verificado: el artefacto ya está en los datos publicados por OpenData (no lo introduce el ETL). No verificado: si nace en Datadis (la distribuidora) o en la agregación del Ajuntament, distinción que requeriría datos a nivel de CUPS no disponibles en abierto. Queda como limitación.

### Los tres patrones
 - Alto a bajo (colapso): el CP perdió sus grandes suministros (08011, 08009, 08007).
 - Bajo a alto (inflación): el CP gana nivel de golpe (consumo reasignado o reportado de más): 08036, 08006, 08026.
 - Pico sostenido de un mes (08009 oct-2023 = 4×): error de acumulación o corrección de lecturas.

### Frases para la memoria
> *"El consumo agregado por código postal presenta saltos de nivel abruptos en 2024-2025, compatibles con la reasignación de grandes puntos de suministro (CUPS) entre códigos o con un error de agregación en el paso de CUPS a código postal. La conservación solo parcial y el declive neto observados son más consistentes con que parte del consumo se pierde en la agregación que con una reasignación limpia. El artefacto está en los datos publicados por OpenData (verificado contra los crudos), no lo introduce el ETL; distinguir su origen exacto (Datadis o agregación municipal) requeriría datos a nivel de CUPS no disponibles en abierto."*

> *"En 2025 se observa además un declive neto (~10%) del consumo reportado, concentrado en el Eixample central. El patrón (pérdidas agrupadas y saltos abruptos) es más consistente con un cambio de atribución o reporte que con variación de demanda real, sin descartar un componente de declive real."*

> *"Una mitigación de futuro sería agregar a nivel de distrito, donde las reasignaciones entre códigos contiguos se cancelan."*